In [64]:
import requests
import pandas as pd

In [65]:
url = " https://services-ap1.arcgis.com/P744lA0wf4LlBZ84/arcgis/rest/services/Vicmap_Features_of_Interest/FeatureServer/1/query"

params = {
    "where": "feature_type='hospital' AND state='VIC'",
    "returnGeometry":"True",
    "f": "json",
    "resultRecordCount": 2000
}

response = requests.get(url, params=params)
data = response.json()

features = data.get('features',[])

hospitals = []

for f in features:
    attrs = f['attributes']
    geom = f.get('geometry', {})
    attrs['x']=geom.get('x')
    attrs['y']=geom.get('y')
    hospitals.append(attrs)


df_hospitals = pd.DataFrame(hospitals)




In [66]:
print(df_hospitals.head())
print(len(df_hospitals))

                                          name             x             y
0                      COBURG ENDOSCOPY CENTRE  1.613741e+07 -4.543466e+06
1            AVIVE CLINIC MORNINGTON PENINSULA  1.615195e+07 -4.604698e+06
2                BALLARAT DAY PROCEDURE CENTRE  1.601134e+07 -4.514721e+06
3  BAYSIDE DAY PROCEDURE AND SPECIALIST CENTRE  1.615729e+07 -4.600379e+06
4                            KNOX DAY HOSPITAL  1.617060e+07 -4.557113e+06
346


In [67]:
import geopandas as gpd
from shapely.geometry import Point

gdf_hospitals = gpd.GeoDataFrame(
    df_hospitals,
    geometry=gpd.points_from_xy(df_hospitals['x'], df_hospitals['y']),
    crs="EPSG:3857"
)

gdf_hospitals = gdf_hospitals.to_crs("EPSG:4326")

gdf_hospitals['lon'] = gdf_hospitals.geometry.x
gdf_hospitals['lat'] = gdf_hospitals.geometry.y

In [68]:
gdf_hospitals

,name,x,y,geometry,lon,lat
0,COBURG ENDOSCOPY CENTRE,1.613741e+07,-4.543466e+06,POINT (144.96478 -37.745),144.964777,-37.745004
1,AVIVE CLINIC MORNINGTON PENINSULA,1.615195e+07,-4.604698e+06,POINT (145.09547 -38.17868),145.095469,-38.178679
2,BALLARAT DAY PROCEDURE CENTRE,1.601134e+07,-4.514721e+06,POINT (143.83229 -37.54054),143.832292,-37.540539
3,BAYSIDE DAY PROCEDURE AND SPECIALIST CENTRE,1.615729e+07,-4.600379e+06,POINT (145.14339 -38.14818),145.143385,-38.148175
4,KNOX DAY HOSPITAL,1.617060e+07,-4.557113e+06,POINT (145.26296 -37.84188),145.262963,-37.841884
...,...,...,...,...,...,...
341,SANDRINGHAM HOSPITAL,1.614339e+07,-4.573916e+06,POINT (145.01851 -37.96099),145.018506,-37.960987
342,MALLEE TRACK - SEA LAKE,1.590285e+07,-4.232105e+06,POINT (142.85771 -35.50049),142.857707,-35.500487
343,SIR JOHN MONASH PRIVATE HOSPITAL,1.615484e+07,-4.567935e+06,POINT (145.12136 -37.91861),145.121361,-37.918615
344,THOMAS EMBLING HOSPITAL,1.614274e+07,-4.549639e+06,POINT (145.01273 -37.78884),145.012727,-37.788843


In [69]:
from shapely.geometry import shape

url = "https://services-ap1.arcgis.com/P744lA0wf4LlBZ84/ArcGIS/rest/services/Vicmap_Admin/FeatureServer/11/query"

all_features = []
batch_size = 2000
offset = 0

while True:
    params = {
        "where": "1=1",
        "outFields":"GAZETTED_LOCALITY_NAME,LOCALITY_NAME",
        "f": "geojson",
        "returnGeometry": "true",
        "resultRecordCount": batch_size,
        "resultOffset": offset
    }

    response = requests.get(url, params=params)
    data = response.json()
    features = data.get('features',[])

    if not features:
        break

    all_features.extend(features)
    offset += batch_size

suburbs_gdf = gpd.GeoDataFrame.from_features(all_features, crs="EPSG:4326")

In [70]:
suburbs_gdf

,geometry,gazetted_locality_name,locality_name
0,"POLYGON ((142.25045 -36.68201, 142.25043 -36.6...",DOOEN,DOOEN
1,"POLYGON ((141.98003 -36.63234, 141.98084 -36.6...",PIMPINIO,PIMPINIO
2,"POLYGON ((142.16271 -36.76658, 142.16119 -36.7...",HAVEN,HAVEN
3,"POLYGON ((144.9866 -37.92319, 144.9865 -37.922...",BRIGHTON,BRIGHTON
4,"POLYGON ((143.87075 -35.49319, 143.86939 -35.4...",MURRABIT WEST,MURRABIT WEST
...,...,...,...
2968,"POLYGON ((145.31596 -37.70112, 145.31607 -37.7...",CHRISTMAS HILLS,CHRISTMAS HILLS
2969,"POLYGON ((145.40036 -37.67188, 145.40022 -37.6...",YARRA GLEN,YARRA GLEN
2970,"POLYGON ((145.0566 -37.73521, 145.05548 -37.73...",HEIDELBERG WEST,HEIDELBERG WEST
2971,"POLYGON ((145.00146 -37.72995, 145.00116 -37.7...",RESERVOIR,RESERVOIR


In [71]:
hospitals_in_suburbs = gpd.sjoin(
    gdf_hospitals,
    suburbs_gdf[['locality_name', 'geometry']],
    how='left',
    predicate='within'
)

In [72]:
hospitals_in_suburbs = hospitals_in_suburbs.drop(columns=['index_right'])
hospitals_in_suburbs

,name,x,y,geometry,lon,lat,locality_name
0,COBURG ENDOSCOPY CENTRE,1.613741e+07,-4.543466e+06,POINT (144.96478 -37.745),144.964777,-37.745004,COBURG
1,AVIVE CLINIC MORNINGTON PENINSULA,1.615195e+07,-4.604698e+06,POINT (145.09547 -38.17868),145.095469,-38.178679,MOUNT ELIZA
2,BALLARAT DAY PROCEDURE CENTRE,1.601134e+07,-4.514721e+06,POINT (143.83229 -37.54054),143.832292,-37.540539,WENDOUREE
3,BAYSIDE DAY PROCEDURE AND SPECIALIST CENTRE,1.615729e+07,-4.600379e+06,POINT (145.14339 -38.14818),145.143385,-38.148175,FRANKSTON
4,KNOX DAY HOSPITAL,1.617060e+07,-4.557113e+06,POINT (145.26296 -37.84188),145.262963,-37.841884,BAYSWATER
...,...,...,...,...,...,...,...
341,SANDRINGHAM HOSPITAL,1.614339e+07,-4.573916e+06,POINT (145.01851 -37.96099),145.018506,-37.960987,SANDRINGHAM
342,MALLEE TRACK - SEA LAKE,1.590285e+07,-4.232105e+06,POINT (142.85771 -35.50049),142.857707,-35.500487,SEA LAKE
343,SIR JOHN MONASH PRIVATE HOSPITAL,1.615484e+07,-4.567935e+06,POINT (145.12136 -37.91861),145.121361,-37.918615,CLAYTON
344,THOMAS EMBLING HOSPITAL,1.614274e+07,-4.549639e+06,POINT (145.01273 -37.78884),145.012727,-37.788843,FAIRFIELD


In [73]:
hospital_counts = hospitals_in_suburbs.groupby('locality_name').size().reset_index(name='num_hospitals')
hospital_counts['locality_name'] = hospital_counts['locality_name'].str.lower()
hospital_counts

,locality_name,num_hospitals
0,alexandra,1
1,apollo bay,1
2,ararat,1
3,ascot vale,1
4,ashwood,1
...,...,...
207,wonthaggi,1
208,wycheproof,1
209,yarram,1
210,yarrawonga,1


In [74]:
hospital_counts.to_csv('hospital_per_suburb.csv', index=False)